This notebook demonstrates the use of the `analyse_moves` function which gives an overview of the scoring consequences of all moves from a given position

Specifically, it gives an output that shows for each move from a given position:
- what the relative score (with optimal) play following that move is compared to the optimal score
- what the impact to each player's score is, ie: does it add more points to your hand or take away points from the opponent

This is intended to help develop an understanding of what good strategy looks like, whether to focus on offensive or defensive play

Note that the comparison here is entirely based on the average point totals, not the win/draw/lose chances

In [ ]:
from cardgame import Game, analyse_moves
import matplotlib.pylab as plt
import seaborn as sns
from operator import itemgetter

sns.set_style("darkgrid")

In [ ]:
# find a position with 14 cards remaining where 5 of them are facedown
# This is a computable position which still has quite high variability in the outcome
target_cards_remaining = 14
target_fd_cards_remaining = 3
while True:
    game = Game.deal()
    while game.legal_moves:
        game = game.random_move()
    moves_to_undo = max(0, len(game.moves) - 36 + target_cards_remaining)
    if not moves_to_undo:
        continue
    if len(game.undo(moves_to_undo).board.facedown_cards) == target_fd_cards_remaining:
        break
evaluation_state = game.undo(moves_to_undo)
evaluation_state

In [ ]:
evaluation_state = Game.load(
    "A♥6♣2♦5♠2♠5♦/3♦8♣8♠2♣4♠5♣/A♦7♣4♥6♥3♣K♠/4♣3♥5♥??2♥7♠/A♣??6♠6♦A♠8♥/K♦7♦4♦K♣K♥8♦//3♠7♥//65432834009910315968168"
)
evaluation_state

In [ ]:
def display_analysis(game, game_analysis):
    ordered_moves = sorted(
        game_analysis.items(), key=lambda k: k[1]["combined"], reverse=True
    )
    (row, col), best_move = ordered_moves[0]
    print(f"The best move is to take the {best_move['card']} at R{row}C{col}")
    print(
        f"The player to move (P{'2' if len(game.moves) % 2 else '1'}) scores an average of {best_move['player_mean']:.2f}"
    )
    print(f"The opponent scores an average of {best_move['opponent_mean']:.2f}")
    for (row, col), move_details in ordered_moves[1:]:
        print()
        print(
            f"Taking the {move_details['card']} at R{row}C{col} is worse than the best move by an average of {move_details['combined']:.2f}"
        )
        print(
            f"The player to move scores {move_details['defensive']:.2f} points {'more' if move_details['defensive'] > 0 else 'less'}"
        )
        print(
            f"The opponent scores {move_details['offensive']:.2f} points {'more' if move_details['offensive'] > 0 else 'less'}"
        )


def plot_analysis(analysis):
    fig, ax = plt.subplots(figsize=(10, 4))
    best_move = max(analysis.values(), key=itemgetter("combined"))
    for marker, data in analysis.items():
        off = data["offensive"]
        defv = data["defensive"]
        comb = data["combined"]
        x = off + defv
        y = comb
        color = "red" if (comb == 0) else "royalblue"
        size = 120 if (comb == 0) else 60
        ax.scatter(x, y, c=color, s=size, zorder=5)
        if (comb != 0) or (x != 0):
            card_str = str(data["card"])
            if card_str == "??":
                card_str = f"FD R{marker[0]}C{marker[1]}"
            ax.annotate(
                card_str, (x, y), textcoords="offset points", xytext=(5, 5), fontsize=12
            )
    ax.text(
        0,
        0.3,
        f"Best move - {str(best_move['card'])}",
        ha="center",
        fontsize=12,
        fontweight="bold",
    )
    ax.set_ylabel("Evaluation difference")
    ax.set_xlabel(
        "<- Lower point totals (offensive)          Symmetric impact         Higher point totals (defensive)->"
    )
    curr = max(abs(bound) for bound in ax.get_xlim())
    ax.set_xlim(-curr, curr)
    plt.grid(axis="x")
    plt.tight_layout()
    plt.show()

In [ ]:
analysis = analyse_moves(evaluation_state)

A move's offensive and defensive character is about if tends to result in more or less points across both players hands
- A move that reduces your opponents score while potentially also reducing your own points is more of an offensive move
- A move that increases your own score while allowing your opponent to also increase theirs is a more defensive move

The only reference point we have here is taking the change in outcomes by making a certain move as compared to the best move in a given position, so everything is measured from that

We can then use this to understand a player's tendencies - if they overprioritise preventing an opponent from forming runs/sets or need to focus on that more

This can help to understand how important a move is
- if all the moves are close together, near the best move at the origin, it tells you it's not an important move
- if there is one move which is substantially worse than the others
- if it's a pivotal move where there is only one good option

In [ ]:
display_analysis(evaluation_state, analysis)

In [ ]:
plot_analysis(analysis)

In [ ]:
evaluation_state.evaluate()